# Equipment replacement as a shortest-path problem

Worked example for ABW Lecture 5 (Exercise 3.9), by Joaquim Gromicho.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/networks/equipment-replacement.ipynb)
[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/networks/equipment-replacement.ipynb)

This notebook implements the equipment-replacement example from the lecture using NetworkX. It was migrated from the [original public Colab notebook](https://colab.research.google.com/drive/1UNhmLYe3aMXPoph02qRHX0f-lZV15b8h); the four calculation cells are unchanged.


## From a replacement schedule to a path

We need a tractor for a three-year planning horizon. Node $i$ represents time $i$, measured in years from the start. An arc $(i,j)$ means acquiring a tractor at time $i$ and keeping it until time $j$.

The arc weights are the net discounted ownership costs of those intervals, in thousands of dollars, as given in the lecture. They include acquisition, running and maintenance costs, less the trade-in value. At an intermediate node, we replace the tractor; node 3 marks the end of the planning horizon.

A path from node 0 to node 3 therefore represents a complete replacement schedule. Its cost is the sum of its arc weights. Before running the notebook, predict which schedule will be cheapest.


## Prepare Python

You can run this notebook in Colab, Binder, or a local Jupyter environment. The next cell uses the course's shared setup helper to install NetworkX and Matplotlib only if they are missing. Run the cells in order.


In [ ]:
# Use installed packages, install only missing ones, without version pins.
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'networkx': 'networkx', 'matplotlib': 'matplotlib'}
ensure_packages(required_packages)


## Define a directed, weighted graph

Each `length` value is an ownership-interval cost, in thousands of dollars.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
g = nx.DiGraph()
g.add_edge(0, 1,length = 8)
g.add_edge(0, 2,length = 18)
g.add_edge(0, 3,length = 31)
g.add_edge(1, 2,length = 10)
g.add_edge(1, 3,length = 21)
g.add_edge(2, 3,length = 12)

## Visualize the graph

The arrows show the available ownership intervals; their labels show the corresponding costs.


In [ ]:
pos = nx.circular_layout(g)
nx.draw(g, pos, with_labels=True)
labels = nx.get_edge_attributes(g,'length')
_= nx.draw_networkx_edge_labels(g, pos, edge_labels=labels, label_pos=0.25)
# plt.savefig('SimpleGraph.pdf', bbox_inches='tight', pad_inches=0)

## Compute a shortest path

Use `length` as the weight so that the algorithm minimizes total cost.


In [ ]:
path = nx.shortest_path(g,source=0,target=3,weight='length')
path

## Compute the total cost

Add the weights of consecutive arcs on the chosen path.


In [ ]:
sum( g[i][j]['length'] for i,j in zip(path[:-1],path[1:]) )

## Interpret the result

The shortest path is `[0, 1, 3]`, with total cost $8 + 21 = 29$ thousand dollars. Acquire a tractor at the start, replace it after year 1, and keep the replacement until the end of year 3.

For comparison, keeping the first tractor for all three years costs 31; replacing only after year 2 costs $18 + 12 = 30$; replacing after both years 1 and 2 costs $8 + 10 + 12 = 30$.

Which arc weights would change if maintenance became more expensive? Predict how this could affect the replacement schedule, then change the graph and run the cells again.
